# 00 — Predictive Representation Geometry

**Engineering question**

Why does next-token prediction reshape transformer hidden states into straighter sentence trajectories?

This notebook is the anchor for the repo. It turns the paper-level result into a compact executable roadmap:

```text
Sentence tokens
→ Transformer hidden states
→ Sentence trajectory
→ Displacement vectors
→ Curvature metric
→ Layerwise straightening
→ Predictive extrapolation
→ Geometric intervention
```

The point is not only that a trained language model predicts the next token.

The point is that training appears to organize internal representation space so that future hidden states become easier to extrapolate from current hidden-state motion.


## Setup

The notebook is designed to run either:

- from the repo root, or
- from inside the `notebooks/` directory.

It creates:

```text
figures/
results/csv/
results/json/
results/00_outputs.zip
```


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Scaling from tokens to hidden trajectories

The paper studies autoregressive transformers trained on next-word prediction.

Instead of asking only whether the output token is correct, it asks a geometric question:

```text
How does the sentence move through hidden-state space as each token enters the model?
```

A sentence becomes a sequence of internal states:

\[
x_1^\ell, x_2^\ell, \ldots, x_n^\ell
\]

where \(x_i^\ell\) is the hidden representation of token \(i\) at layer \(\ell\).

The first figure states the representational transition.


In [ ]:
steps = [
    "Sentence\ntokens",
    "Transformer\nlayers",
    "Hidden-state\nsequence",
    "Sentence\ntrajectory",
    "Prediction\ngeometry",
]

x = np.arange(len(steps))

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.scatter(x, np.zeros_like(x), s=260, zorder=3)

for i in range(len(x) - 1):
    ax.annotate(
        "",
        xy=(x[i + 1], 0),
        xytext=(x[i], 0),
        arrowprops=dict(arrowstyle="->", linewidth=2.2),
    )

for xi, label in zip(x, steps):
    ax.text(xi, -0.18, label, ha="center", va="top", fontsize=11)

ax.set_title("From Tokens to Predictive Representation Geometry", fontsize=16)
ax.set_xlim(-0.4, len(steps) - 0.6)
ax.set_ylim(-0.55, 0.45)
ax.axis("off")

savefig(fig, "00_tokens_to_geometry.png")
plt.show()


## 2. Curvature as the leading measurement

A sentence trajectory is represented as a polygonal chain through hidden-state space.

Adjacent displacement vectors are:

\[
v_i^\ell = x_{i+1}^\ell - x_i^\ell
\]

Curvature is estimated from the angle between adjacent displacement vectors:

\[
c_i^\ell =
\arccos
\left(
\frac{v_i^\ell \cdot v_{i+1}^\ell}
{\|v_i^\ell\| \, \|v_{i+1}^\ell\|}
\right)
\]

A straighter trajectory has smaller average angle.

This figure illustrates the difference between high-curvature and low-curvature token paths.


In [ ]:
# Toy 2D trajectories for visual intuition only.
t = np.arange(8)
high_x = t
high_y = np.array([0.0, 1.0, -0.3, 1.3, 0.1, 1.1, -0.2, 0.7])

low_x = t
low_y = 0.16 * t + 0.12 * np.sin(t / 1.5)

fig, ax = plt.subplots(figsize=(9, 4.8))

ax.plot(high_x, high_y, marker="o", linewidth=2.2, label="Early layer: high curvature")
ax.plot(low_x, low_y - 1.8, marker="s", linewidth=2.2, label="Middle layer: lower curvature")

for i in range(len(t)):
    ax.text(high_x[i], high_y[i] + 0.12, f"w{i+1}", ha="center", fontsize=9)
    ax.text(low_x[i], low_y[i] - 1.62, f"w{i+1}", ha="center", fontsize=9)

ax.set_title("Sentence Trajectory Straightening", fontsize=16)
ax.set_xlabel("Token order")
ax.set_ylabel("Hidden-state projection")
ax.grid(True, alpha=0.25)
ax.legend()

savefig(fig, "00_sentence_trajectory_straightening.png")
plt.show()


## 3. Layerwise straightening profile

The paper reports a characteristic pattern:

```text
early layers → middle layers: curvature decreases
later layers: curvature may rise again near output
```

That shape matters.

The strongest geometry is not necessarily the final layer. The middle layers are where the model appears to construct the most predictive representation.


In [ ]:
layers = np.arange(0, 49)

# Synthetic profile for context notebook only.
drop = -11.5 * (1 - np.exp(-layers / 8))
late_rise = 5.5 / (1 + np.exp(-(layers - 32) / 3.5))
trained_profile = drop + late_rise

untrained_profile = 0.35 * np.sin(layers / 3.0)

fig, ax = plt.subplots(figsize=(8.8, 4.8))

ax.plot(layers, trained_profile, marker="o", markersize=3.5, label="Trained model profile")
ax.plot(layers, untrained_profile, marker="s", markersize=3.5, label="Untrained reference")

ax.axhline(0, linewidth=1)
ax.set_title("Expected Curvature Change Across Transformer Depth", fontsize=16)
ax.set_xlabel("Layer")
ax.set_ylabel("Curvature change relative to first layer")
ax.grid(True, alpha=0.3)
ax.legend()

savefig(fig, "00_layerwise_curvature_profile.png")
plt.show()


## 4. Measurement-to-intervention roadmap

The original paper measures an emergent geometric property.

This repo asks the next-step engineering question:

```text
Can predictive representation geometry be deliberately shaped without damaging language prediction?
```

The first intervention is a cosine-alignment gate.

For adjacent displacement vectors,

\[
v_i^\ell = x_{i+1}^\ell - x_i^\ell
\]

we can encourage alignment with:

\[
\mathcal{L}_{align}
=
1 -
\frac{v_i^\ell \cdot v_{i+1}^\ell}
{\|v_i^\ell\| \, \|v_{i+1}^\ell\|}
\]

The total training objective becomes:

\[
\mathcal{L}
=
\mathcal{L}_{NTP}
+
\lambda \mathcal{L}_{align}
\]


In [ ]:
roadmap = [
    "Observe\nstraightening",
    "Measure\ncurvature",
    "Reproduce\nlayer profile",
    "Add cosine\nalignment",
    "Add curvature\nregularizer",
    "Compare\ngeometry + loss",
    "Generalize\nacross scale",
]

x = np.arange(len(roadmap))

fig, ax = plt.subplots(figsize=(12.5, 3.5))
ax.scatter(x, np.zeros_like(x), s=190, zorder=3)

for i in range(len(x) - 1):
    ax.annotate(
        "",
        xy=(x[i + 1], 0),
        xytext=(x[i], 0),
        arrowprops=dict(arrowstyle="->", linewidth=1.9),
    )

for xi, label in zip(x, roadmap):
    ax.text(xi, -0.18, label, ha="center", va="top", fontsize=9.5)

ax.set_title("Research Roadmap: From Observation to Geometric Intervention", fontsize=16)
ax.set_xlim(-0.4, len(roadmap) - 0.6)
ax.set_ylim(-0.55, 0.45)
ax.axis("off")

savefig(fig, "00_research_roadmap.png")
plt.show()


## 5. Hypotheses

This repo should not stop at reproducing a plot.

It should test whether trajectory straightening is only correlated with next-token prediction or whether it can be used as an intervention.

The table below is the specification scaffold for the rest of the notebooks.


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Stage": "Context",
            "Focus": "Predictive representation geometry",
            "Question": "Why measure hidden-state trajectories instead of only output loss?",
            "Next notebook": "00_context",
        },
        {
            "Stage": "Sentence trajectories",
            "Focus": "Hidden states by token and layer",
            "Question": "How do sentence tokens trace paths through model representation space?",
            "Next notebook": "07_sentence_trajectories",
        },
        {
            "Stage": "Curvature",
            "Focus": "Angle between adjacent displacement vectors",
            "Question": "Can we reproduce the trained-vs-untrained curvature profile?",
            "Next notebook": "13_curvature",
        },
        {
            "Stage": "Cosine alignment",
            "Focus": "Alignment of adjacent displacement vectors",
            "Question": "Does an explicit alignment objective amplify useful straightening?",
            "Next notebook": "17_cosine_alignment",
        },
        {
            "Stage": "Curvature regularization",
            "Focus": "Direct penalty on trajectory curvature",
            "Question": "Does curvature regularization improve prediction or make language brittle?",
            "Next notebook": "23_curvature_regularization",
        },
        {
            "Stage": "Ablations",
            "Focus": "Loss, surprisal, curvature, generated continuations",
            "Question": "Which geometric intervention improves prediction without over-flattening?",
            "Next notebook": "29_ablations",
        },
        {
            "Stage": "Intrinsic dimension",
            "Focus": "Layerwise dimensionality and manifold structure",
            "Question": "Does straightening co-evolve with lower intrinsic dimension?",
            "Next notebook": "37_intrinsic_dimension",
        },
        {
            "Stage": "Scaling",
            "Focus": "Model size, data scale, regularization strength",
            "Question": "Do small models benefit more from explicit geometric constraints?",
            "Next notebook": "43_scaling",
        },
        {
            "Stage": "Future directions",
            "Focus": "Predictive geometry beyond GPT-style language models",
            "Question": "Where else does temporal prediction induce trajectory straightening?",
            "Next notebook": "49_future_directions",
        },
    ]
)

summary.to_csv(CSV / "00_summary.csv", index=False)
summary.to_json(JS / "00_summary.json", orient="records", indent=2)

summary


## 6. Compact interpretation

The paper reframes next-token prediction as a hidden-state geometry problem.

A trained autoregressive transformer does not only map context to token probabilities.

It also reshapes sentence trajectories so that internal states move through representation space in a more linearly extrapolatable way.

That gives the repo its next-step question:

```text
If training naturally straightens sentence trajectories,
can we engineer that geometry directly?
```

The strongest first test is modest mid-layer cosine alignment, compared against a baseline and a direct curvature regularizer.


## 7. Export and download

This cell packages the outputs.

It uses `google.colab.files.download()` when running in Colab, and falls back to an IPython `FileLink` in local Jupyter.

This download pattern should be reused in the next notebooks.


In [ ]:
zip_path = RES / "00_outputs.zip"

files_to_zip = [
    FIG / "00_tokens_to_geometry.png",
    FIG / "00_sentence_trajectory_straightening.png",
    FIG / "00_layerwise_curvature_profile.png",
    FIG / "00_research_roadmap.png",
    CSV / "00_summary.csv",
    JS / "00_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Next notebook

**07 — Sentence Trajectories**

The next notebook should compute hidden-state trajectories from a small transformer language model.

Candidate outputs:

```text
07_hidden_state_shapes.png
07_token_trajectory_projection.png
07_layerwise_trajectory_grid.png
07_sentence_batch_metadata.csv
```

Core question:

```text
How do sentence tokens trace paths through representation space across transformer layers?
```
